# Model Comparison: Linear Regression vs Random Forest

Research Question: To what extent can machine learning models predict airline arrival delays using pre-departure information?

## Problem Statement

- Mục tiêu: so sánh performance giữa `LinearRegression` và `RandomForestRegressor` trên cùng một dataset.
- Dữ liệu: `data/processed_flight_data.csv`.
- Các chỉ số đánh giá: MAE, RMSE, R².
- Kết quả cần đưa ra bảng so sánh và biểu đồ trực quan.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

data_path = Path('..') / 'data' / 'processed_flight_data.csv'
image_path = Path('..') / 'images'
image_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)
print('Loaded dataset with shape:', df.shape)
print('Target column present:', 'ArrDelay' in df.columns)

## Data and Features

- Sử dụng các feature có sẵn trước khi cất cánh.
- Chúng ta giữ các biến vận hành và lịch trình đã được xử lý.

In [ ]:
selected_features = [
    'DepDelay',
    'CRSElapsedTime',
    'Distance',
    'DepHour',
    'IsWeekend',
    'IsRushHour',
    'Origin_Freq',
    'Dest_Freq',
    'Month',
    'DayOfWeek'
]
target_column = 'ArrDelay'

required_columns = selected_features + [target_column]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

model_df = df[required_columns].dropna()
print('Model dataset shape after dropna:', model_df.shape)
model_df[selected_features].head()

## Train/Test Split

- Chia dữ liệu 75/25 để so sánh mô hình với một test set chung.

In [ ]:
X = model_df[selected_features]
y = model_df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

## Feature Scaling for Linear Regression

- Random Forest không cần scaling, nhưng Linear Regression sẽ dùng `StandardScaler` để đảm bảo hiệu quả và so sánh công bằng.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Feature scaling completed.')

## Model Training

- Huấn luyện `LinearRegression` và `RandomForestRegressor` trên cùng tập train.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print('Linear Regression trained')
print('Random Forest trained')

## Evaluation Functions

- Định nghĩa hàm dùng chung cho cả hai mô hình để thu thập MAE, RMSE, R².

In [ ]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

lr_pred = lr_model.predict(X_test_scaled)
rf_pred = rf_model.predict(X_test)

lr_metrics = evaluate('Linear Regression', y_test, lr_pred)
rf_metrics = evaluate('Random Forest', y_test, rf_pred)

comparison_df = pd.DataFrame([lr_metrics, rf_metrics])
comparison_df['MAE'] = comparison_df['MAE'].map(lambda x: round(x, 3))
comparison_df['RMSE'] = comparison_df['RMSE'].map(lambda x: round(x, 3))
comparison_df['R2'] = comparison_df['R2'].map(lambda x: round(x, 4))
comparison_df

## Comparison Table

- Bảng này trình bày so sánh trực tiếp giữa hai mô hình.

In [ ]:
comparison_df

## Visualization

- So sánh RMSE và R² giữa hai mô hình bằng biểu đồ cột trực quan.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison_df['Model'], comparison_df['RMSE'], color=['#4C72B0', '#55A868'])
plt.title('RMSE Comparison')
plt.ylabel('RMSE')
plt.grid(axis='y', alpha=0.3)
rmse_path = image_path / 'model_comparison_rmse.png'
plt.tight_layout()
plt.savefig(rmse_path, dpi=200)
plt.show()
print('Saved RMSE comparison plot to', rmse_path)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison_df['Model'], comparison_df['R2'], color=['#4C72B0', '#55A868'])
plt.title('R2 Comparison')
plt.ylabel('R2')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
r2_path = image_path / 'model_comparison_r2.png'
plt.tight_layout()
plt.savefig(r2_path, dpi=200)
plt.show()
print('Saved R2 comparison plot to', r2_path)

## Best Model and Tradeoffs

- Dựa trên chỉ số RMSE và R², ta chọn mô hình tốt nhất.
- Giải thích tradeoff giữa interpretability, performance và training time.

In [ ]:
best_model = comparison_df.loc[comparison_df['RMSE'].idxmin(), 'Model']
print('Best model on RMSE:', best_model)
print('
Model comparison summary:')
print(comparison_df.to_string(index=False))

### Interpretability vs Performance

- `Linear Regression` có ưu thế về interpretability vì hệ số tuyến tính dễ giải thích.
- `Random Forest` thường có performance tốt hơn với dữ liệu phi tuyến nhưng khó giải thích hơn.
- Training time: Random Forest mất nhiều thời gian hơn do nhiều cây, trong khi Linear Regression rất nhanh.

### Conclusion

- Chọn best model dựa trên RMSE và R².
- Nếu hiệu suất là ưu tiên chính, mô hình thường tốt hơn là Random Forest; nếu giải thích rõ ràng là quan trọng, Linear Regression vẫn có giá trị.